# Daily readiness

A morning glance at how ready I am to train, built from my intervals.icu data.

It compares my recent HRV, resting heart rate and sleep with my own normal, shows acute vs chronic training load, classifies yesterday's training, and gives simple rule-based guidance for today.

**To look at a past day**, change `SELECTED_DATE` in the setup cell and run all cells.

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go

from IPython.display import HTML, display
from hll.config import PROCESSED_DIR

# Day to show. None means "latest day that has data". Set e.g. "2026-10-18" to look back.
SELECTED_DATE = None

### Settings and thresholds

Everything I might want to tweak lives here. The thresholds are my own starting heuristics, not validated cut-offs.

- **Baseline** = my average over the last 10 days.
- **Normal range** = my 60-day average ± 0.75 standard deviations.
- HRV is **Watch** when the baseline drops below the normal range, and **Flag** when it is 1.5 SDs below the 60-day average.

In [ ]:
# Window lengths in days
BASELINE_DAYS = 10     # recent "baseline"
NORMAL_DAYS = 60       # longer-term "normal"
CV_DAYS = 7            # window for HRV variability

# Readiness thresholds
HRV_WATCH_SD, HRV_FLAG_SD = 0.75, 1.5      # SDs below the 60-day mean (normal range = mean ± HRV_WATCH_SD)
CV_WATCH, CV_FLAG = 1.25, 1.5              # multiples of my typical CV
RHR_WATCH_BPM, RHR_FLAG_BPM = 2, 4         # bpm above normal
SLEEP_WATCH_H, SLEEP_FLAG_H = 0.75, 1.5    # hours below my 7-night average
AC_LOW, AC_HIGH, AC_FLAG = 0.8, 1.3, 1.5   # acute:chronic bands

# Easy / hard cut-offs: percentiles of my own training days
EASY_PCT, HARD_PCT = 0.33, 0.67

# Guidance: each signal adds points, and the total decides green / amber / red
POINTS = {"OK": 0, "Watch": 1, "Flag": 2, "No data": 0}
AMBER_POINTS, RED_POINTS = 2, 4

### Load wellness data anc check which fields are useable
Load data, sort and index by date.
Show proportion of days where values are missing - subjective markers lack sufficient data

In [ ]:
well = pd.read_csv(PROCESSED_DIR / "wellness.csv", parse_dates=["date"])
well = well.sort_values("date").set_index("date")

print("rows", len(well), "| range", well.index.min().date(), "to", well.index.max().date())

well.isna().mean().round(2)

### Prepare the data

Add a row for any missing calendar day, so that a "10-day" window always means 10 days. Convert sleep from seconds to hours.

In [ ]:
well = well.asfreq("D")
well["sleep_h"] = well["sleep_secs"] / 3600

### Choose the day

By default this is the latest day with an HRV value, so if today's data hasn't synced yet I see the most recent day that has it.

In [ ]:
# last_valid_index() is the date of the last row with an HRV value, so if today's
# data hasn't synced yet we show the most recent day that has it.
if SELECTED_DATE is None:
    day = well["hrv"].last_valid_index()
else:
    day = pd.Timestamp(SELECTED_DATE)

print("Showing:", day.date())

### Helper: slice a window

Returns the last `n_days` of a series, up to and including the chosen day. Every baseline below uses it.

In [ ]:
def window(series, end_day, n_days):
    """The last n_days of a series, up to and including end_day."""
    return series.loc[end_day - pd.Timedelta(days=n_days - 1): end_day]

### Baselines and normal ranges

- **HRV:** the 10-day baseline, and the 60-day normal range (mean ± 0.75 SD).
- **HRV variability:** the coefficient of variation (SD as a % of the mean) over 7 days, against my typical 7-day CV.
- **Resting HR:** the 10-day baseline against the 60-day normal.
- **Sleep:** last night against the 7 nights before it.

In [ ]:
# HRV: 10-day baseline against the 60-day normal range
hrv_baseline = window(well["hrv"], day, BASELINE_DAYS).mean()
hrv_normal = window(well["hrv"], day, NORMAL_DAYS).mean()
hrv_normal_sd = window(well["hrv"], day, NORMAL_DAYS).std()
hrv_low = hrv_normal - HRV_WATCH_SD * hrv_normal_sd
hrv_high = hrv_normal + HRV_WATCH_SD * hrv_normal_sd
hrv_last_night = well.loc[day, "hrv"]

# HRV variability: 7-day CV, against the 7-day CV I usually have
hrv_cv_window = window(well["hrv"], day, CV_DAYS)
hrv_cv = 100 * hrv_cv_window.std() / hrv_cv_window.mean()
cv_by_day = 100 * well["hrv"].rolling(CV_DAYS).std() / well["hrv"].rolling(CV_DAYS).mean()
hrv_cv_normal = window(cv_by_day, day, NORMAL_DAYS).mean()

# Resting HR
rhr_baseline = window(well["resting_hr"], day, BASELINE_DAYS).mean()
rhr_normal = window(well["resting_hr"], day, NORMAL_DAYS).mean()

# Sleep: last night against the 7 nights before it
sleep_last_night = well.loc[day, "sleep_h"]
sleep_usual = window(well["sleep_h"], day - pd.Timedelta(days=1), 7).mean()

### Helper: rate a signal

`rate` turns a "how bad is it" number into OK, Watch or Flag. `rate_ratio` does the same for acute:chronic load, where both too high and too low are worth a look.

In [ ]:
def rate(badness, watch, flag):
    """Bigger badness = worse. Returns 'OK', 'Watch', 'Flag' or 'No data'."""
    if pd.isna(badness):
        return "No data"
    if badness >= flag:
        return "Flag"
    if badness >= watch:
        return "Watch"
    return "OK"


def rate_ratio(r):
    """Acute:chronic. 'Watch' can mean too high or too low."""
    if pd.isna(r):
        return "No data"
    if r >= AC_FLAG:
        return "Flag"
    if r > AC_HIGH or r < AC_LOW:
        return "Watch"
    return "OK"

### Establish comparison rules
Look for drops in HRV, increased coefficient of variation, rises in RHR, and decreased total sleep time 

In [ ]:
hrv_pct = 100 * (hrv_recent / hrv_baseline - 1)          # % vs baseline
hrv_z = (hrv_recent - hrv_baseline) / hrv_baseline_sd    # distance from baseline in SDs
cv_ratio = hrv_cv / hrv_cv_baseline                      # 1.0 = typical
rhr_diff = rhr_recent - rhr_baseline                     # bpm vs baseline
sleep_shortfall = sleep_usual - sleep_last_night         # hours less than usual

hrv_status = rate(-hrv_z, HRV_WATCH_SD, HRV_FLAG_SD)     # only drops in HRV are flagged
cv_status = rate(cv_ratio, CV_WATCH, CV_FLAG)
rhr_status = rate(rhr_diff, RHR_WATCH_BPM, RHR_FLAG_BPM)
sleep_status = rate(sleep_shortfall, SLEEP_WATCH_H, SLEEP_FLAG_H)

### Daily training load
Total `icu_load` for each day. Rest days have no activities, so they are added back with zero load. The window stops at the day *before* the chosen morning, because that day's training hasn't happened yet.

In [ ]:
act = pd.read_csv(PROCESSED_DIR / "activities.csv", parse_dates=["start_date"])
act["date"] = act["start_date"].dt.normalize()

daily_load = act.groupby("date")["icu_load"].sum()      # total load per day

# Rest days have no activities, so add them back with zero load. The window stops at the
# day BEFORE the selected morning, because that day's training hasn't happened yet.
load_end = day - pd.Timedelta(days=1)
all_days = pd.date_range(daily_load.index.min(), load_end)
daily_load = daily_load.reindex(all_days, fill_value=0)

### Acute and chronic load

- **Acute** = average daily load over the last 7 days.
- **Chronic** = average daily load over the last 28 days.
- **Ratio** = acute ÷ chronic. Around 1 means I'm training as usual; well above 1 means recent load is high compared with what I'm used to.

In [ ]:
acute = daily_load.rolling(7).mean()       # average daily load, last 7 days
chronic = daily_load.rolling(28).mean()    # average daily load, last 28 days

acute_now = acute.iloc[-1]
chronic_now = chronic.iloc[-1]
ac_ratio = acute_now / chronic_now
ac_status = rate_ratio(ac_ratio)

print("load up to", load_end.date(), "| acute", show(acute_now), "| chronic", show(chronic_now),
      "| ratio", show(ac_ratio, 2))

### Yesterday's training

Classify yesterday as Rest, Easy, Medium or Hard **relative to my own history**: the bottom third of my last 60 training days counts as easy, the top third as hard. Also count how many of the last 3 days were hard.

In [ ]:
# What counts as easy and hard for me: percentiles of my own training days
training_days = daily_load.tail(NORMAL_DAYS)
training_days = training_days[training_days > 0]
EASY_PCT, HARD_PCT = 0.5, 0.85     # hard is now the top ~15% of training days
HARD_MIN_MULTIPLE = 1.3            # and must be at least 1.3x a typical training day
easy_max, hard_min = training_days.quantile([EASY_PCT, HARD_PCT])
hard_min = max(hard_min, HARD_MIN_MULTIPLE * training_days.median())

def classify(load):
    """Rest, Easy, Medium or Hard, compared with my own training days."""
    if load == 0:
        return "Rest"
    if load >= hard_min:
        return "Hard"
    if load > easy_max:
        return "Medium"
    return "Easy"


yesterday_load = daily_load.iloc[-1]
yesterday_class = classify(yesterday_load)
hard_days_last3 = sum(classify(load) == "Hard" for load in daily_load.tail(3))

sessions = act.loc[act["date"] == load_end, "name"].astype(str)
yesterday_sessions = ", ".join(sessions) if len(sessions) else "no training"

print(f"yesterday: {yesterday_class} (load {yesterday_load:.0f}; hard from {hard_min:.0f}) | hard days in last 3: {hard_days_last3}")

### Guidance for today

Simple rules, nothing learned from data. Each signal that is off adds points (Watch = 1, Flag = 2):

- **0–1 points: Green.** Train as planned.
- **2–3 points: Amber.** Keep it easy to moderate.
- **4 or more: Red.** Rest or very easy movement.

Yesterday counts too: a hard day adds 1 point, and 2 or more hard days in the last 3 add another. A *high* acute:chronic ratio adds points, but a *low* one doesn't, because low recent load means room to train, not fatigue. The reasons are collected so the dashboard can say why.

In [ ]:
points = 0
reasons = []

signals = [
    (hrv_status, "HRV baseline is below my normal range"),
    (cv_status, "HRV is more variable than usual"),
    (rhr_status, "resting HR is above normal"),
    (sleep_status, "last night's sleep was short"),
]
for status, reason in signals:
    if status in ("Watch", "Flag"):
        points += POINTS[status]
        reasons.append(reason)

if "No data" in (hrv_status, cv_status, rhr_status, sleep_status):
    reasons.append("some readiness data is missing, so treat this with caution")

if ac_ratio > AC_HIGH:
    points += POINTS[ac_status]
    reasons.append("recent load is high compared with my 4-week average")
elif ac_ratio < AC_LOW:
    reasons.append("recent load is low compared with my 4-week average, so there is room to build")

if yesterday_class == "Hard":
    points += 1
    reasons.append("yesterday was a hard day")
if hard_days_last3 >= 2:
    points += 1
    reasons.append(f"{hard_days_last3} of the last 3 days were hard")

if points >= RED_POINTS:
    guidance = "Red"
elif points >= AMBER_POINTS:
    guidance = "Amber"
else:
    guidance = "Green"

if not reasons:
    reasons = ["every signal is within my normal range"]

ADVICE = {
    "Green": "Train as planned",
    "Amber": "Keep it easy to moderate",
    "Red": "Rest or very easy movement",
}

print(guidance, "| points", points, "|", reasons)

### Display: Colour settings and tile functions

Colours help to improve interpretation.
`show` turns a number into text (a dash if it is missing). `status_tile` builds one HTML tile with a coloured edge, and `tile_row` puts tiles side by side. The styles are named once so the tile function stays short.

In [ ]:
BG, CARD = "#12141f", "#1a1d2e"
INK, MUTED, GRID = "#e8eaf2", "#8b90a7", "#2a2e45"
GOOD, WARN, CRIT = "#0ca30c", "#fab219", "#d03b3b"
BLUE, ORANGE = "#3987e5", "#d95926"

STATUS = {
    "OK": (GOOD, "●"), "Watch": (WARN, "▲"), "Flag": (CRIT, "✖"), "No data": (MUTED, "–"),
    "Green": (GOOD, "●"), "Amber": (WARN, "▲"), "Red": (CRIT, "✖"),
    "Rest": (MUTED, "–"), "Easy": (BLUE, "▁"), "Medium": (BLUE, "▄"), "Hard": (BLUE, "█"),
}

def show(value, decimals=0, sign=False):
    """Number to text; sign=True adds a + for positive numbers. Missing -> dash."""
    if pd.isna(value):
        return "–"
    plus = "+" if sign else ""
    return f"{value:{plus}.{decimals}f}"


TILE = f"flex:1; background:{CARD}; border-radius:8px; padding:16px 14px;"
LABEL = f"color:{MUTED}; font-size:13px;"
VALUE = f"color:{INK}; font-size:32px; font-weight:700;"
STATUS_TEXT = f"color:{INK}; font-size:13px; font-weight:600; margin-top:8px;"


def status_tile(label, value, unit, detail, status):
    """One tile: label, big value, detail line, and a coloured edge with icon and status word."""
    colour, icon = STATUS[status]
    return f"""
    <div style="{TILE} border-left:6px solid {colour};">
      <div style="{LABEL}">{label}</div>
      <div style="{VALUE}">{value} <span style="{LABEL}">{unit}</span></div>
      <div style="{LABEL}">{detail}</div>
      <div style="{STATUS_TEXT}"><span style="color:{colour};">{icon}</span> {status}</div>
    </div>"""


def tile_row(tiles_html):
    """Put tiles side by side."""
    return f'<div style="display:flex; gap:12px; margin-bottom:12px;">{tiles_html}</div>'

### Build tiles
Fill each tile with value, status and some detail

In [ ]:
guidance_text = "<br>".join("• " + r[0].upper() + r[1:] + "." for r in reasons)
guidance_tile = status_tile("Today's guidance", ADVICE[guidance], "", guidance_text, guidance)

readiness_tiles = (
    status_tile("HRV (rMSSD), 10-day baseline", show(hrv_baseline), "ms",
                f"normal range {show(hrv_low)}–{show(hrv_high)} ms (60-day)<br>last night {show(hrv_last_night)} ms",
                hrv_status)
    + status_tile("HRV variability, 7-day CV", show(hrv_cv, 1), "%",
                  f"typical {show(hrv_cv_normal, 1)}% ({show(cv_ratio, 2)}×)", cv_status)
    + status_tile("Resting HR, 10-day baseline", show(rhr_baseline, 1), "bpm",
                  f"{show(rhr_diff, 1, sign=True)} bpm vs 60-day normal ({show(rhr_normal, 1)})", rhr_status)
)

context_tiles = (
    status_tile("Sleep, last night", show(sleep_last_night, 1), "h",
                f"7-night average {show(sleep_usual, 1)} h", sleep_status)
    + status_tile("Acute:chronic load", show(ac_ratio, 2), "",
                  f"acute {show(acute_now)} · chronic {show(chronic_now)} (avg daily load)", ac_status)
    + status_tile("Yesterday's training", yesterday_class, "",
                  f"load {show(yesterday_load)} (hard from {show(hard_min)})<br>{yesterday_sessions}", yesterday_class)
)

### Build chart and dashboard

In [ ]:
recent = pd.DataFrame({"acute": acute, "chronic": chronic}).tail(56)

fig = go.Figure()
fig.add_scatter(x=recent.index, y=recent["acute"], name="Acute (7-day avg)",
                line=dict(color=BLUE, width=2))
fig.add_scatter(x=recent.index, y=recent["chronic"], name="Chronic (28-day avg)",
                line=dict(color=ORANGE, width=2))
fig.update_layout(
    template="plotly_dark", height=320,
    paper_bgcolor=CARD, plot_bgcolor=CARD,
    title=dict(text="Acute vs chronic load", font=dict(size=16, color=INK)),
    legend=dict(orientation="h", y=1.15, x=0),
    margin=dict(t=70, l=50, r=20, b=40),
)
fig.update_xaxes(showgrid=False)
fig.update_yaxes(gridcolor=GRID, title_text="Avg daily load")

display(HTML(f"""
<div style="background:{BG}; padding:16px 16px 4px; border-radius:10px; font-family:sans-serif;">
  <div style="color:{INK}; font-size:18px; margin-bottom:12px;">Readiness · {day:%a %d %b %Y}</div>
  {tile_row(guidance_tile)}
  {tile_row(readiness_tiles)}
  {tile_row(context_tiles)}
</div>"""))

fig.show()


### Limitations and next steps

- **Heuristics, not science.** Every threshold, and the rule-based guidance, is estimated and for demonstration purposes.
- **HRV.** Raw rMSSD from my device, compared with my own history. Only drops are flagged. The 10-day baseline sits inside the 60-day window, so the two overlap and the comparison is slightly damped.
- **Acute:chronic load** is descriptive context, not an injury predictor, and the commonly quoted 0.8–1.3 band is contested. It uses `icu_load` up to the day before, and only a *high* ratio feeds the guidance.
- **Easy, medium and hard** are relative to my own last 60 training days, using the day's total load, so one long easy day can look "hard". `icu_load` primarily depends on heart rate, which has sometimes been unreliable.
- **Sleep** is duration only, compared with my previous 7 nights.
- **Subjective ratings** (soreness, mood, stress and so on) depend on user input, and sufficient data is not yet there.
- **Next:** compare `icu_load` with RPE where heart rate is unreliable, add monotony and strain, and look at mechanical load.